In [ ]:
import pandas as pd

#loading the dataset
df = pd.read_csv('data/spam.csv', encoding='latin-1')

#inspecting the dataset
df.shape
df.info()
df.describe()
df.head(10)
df.tail(10)

#editing the v1 and v2 columns to be more descriptive as 'Label' and 'Message'
df = df.rename(columns={'v1': 'Label', 'v2': 'Message'})
df.head(3)

#inspecting class balance and missing values
df['Label'].value_counts() # count of each class
df['Label'].value_counts(normalize=True) * 100 # percentage of each class
df.isnull().sum() # checking for missing values
df['Label'].unique() # checking for unique values in the 'Label' column

#handling the unamed columns
df[df[['Unnamed: 2','Unnamed: 3','Unnamed: 4']].notna().any(axis=1)]
junk_col = ['Unnamed: 2','Unnamed: 3','Unnamed: 4']
for col in junk_col:
    #turning Nan values into empty strings
    spill = df[col].fillna('')
    # only add a space + the spillover text where it actually has content
    df['Message'] = df['Message'] + spill.apply(lambda s: ' ' + s if s else '')
df = df.drop(columns=junk_col)

#removing duplicates
df.duplicated(subset='Message').sum()
df = df.drop_duplicates(subset=['Message']).reset_index(drop=True)
#checking the number of duplicates removed
df.duplicated(subset='Message').sum()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5572 entries, 0 to 5571
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   v1          5572 non-null   object
 1   v2          5572 non-null   object
 2   Unnamed: 2  50 non-null     object
 3   Unnamed: 3  12 non-null     object
 4   Unnamed: 4  6 non-null      object
dtypes: object(5)
memory usage: 217.8+ KB


np.int64(0)

In [65]:
# writing a cleaned text function to remove any non-alphanumeric characters and convert the text to lowercase
import re 
import string 

def clean_text(text):
    # Remove punctuation
    text = text.translate(str.maketrans('', '', string.punctuation))
    # Remove non-alphanumeric characters
    text = re.sub(r'[^a-zA-Z0-9\s]', '', text)
    # Convert to lowercase
    text = text.lower()
    return text

df['Message'] = df['Message'].apply(clean_text)

#validating the cleaned text
df = df[df['Label'].isin(['spam', 'ham'])]
df = df[df['Message'].str.len() > 0]
df = df.reset_index(drop=True)


In [70]:
from sklearn.model_selection import train_test_split

X = df['Message']
y = df['Label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

Summary of findings

- Raw dataset: 5,572 rows, 2 usable columns (label, text) after fixing spillover columns
- After removing duplicates and invalid rows: X rows remaining
- Class balance: ham X%, spam X% — imbalanced, so Role 3 should rely on
  precision/recall/F1 and the confusion matrix rather than accuracy alone
- Duplicate messages removed: X
- Missing values: 0
- Split: test_size=0.2, random_state=42, stratified — X train / X test rows
- Cleaned split saved to data/train.csv and data/test.csv for Role 2
